# Rabin-Karp — Rolling Hash & Fingerprinting

In [ ]:
%config InlineBackend.figure_format = "retina"

import warnings
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np

warnings.filterwarnings("ignore", message=".*tight_layout.*")


def safe_tight_layout(fig=None):
    """Call tight_layout suppressing the aspect-ratio incompatibility warning."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", UserWarning)
        if fig is None:
            plt.tight_layout()
        else:
            fig.tight_layout()


plt.rcParams.update({
    "font.family": "monospace",
    "font.size": 13,
    "figure.facecolor": "#fafafa",
    "figure.dpi": 150,
    "savefig.dpi": 150,
})

COLORS = {
    "match": "#4CAF50",
    "mismatch": "#F44336",
    "current": "#FFC107",
    "skip": "#90CAF9",
    "default": "#E0E0E0",
    "pattern": "#BBDEFB",
    "border": "#7E57C2",
    "hash_match": "#CE93D8",
    "hash_miss": "#FFCC80",
    "verified": "#66BB6A",
    "spurious": "#EF5350",
}


def draw_string_row(ax, y, string, label="", highlights=None, offset=0):
    """Draw a row of character boxes at vertical position y."""
    highlights = highlights or {}
    for i, ch in enumerate(string):
        color = highlights.get(i, COLORS["default"])
        rect = mpatches.FancyBboxPatch(
            (i + offset, y), 0.9, 0.9,
            boxstyle="round,pad=0.05",
            facecolor=color, edgecolor="#555", linewidth=1.2,
        )
        ax.add_patch(rect)
        ax.text(i + offset + 0.45, y + 0.45, ch,
                ha="center", va="center", fontsize=14, fontweight="bold")
    if label:
        ax.text(offset - 0.3, y + 0.45, label,
                ha="right", va="center", fontsize=12, color="#555")


def make_stepper(step_widget, label="Step"):
    """Create a prev/next/first/last button bar tied to an IntSlider (hidden)."""
    btn_first = widgets.Button(description="|<", layout=widgets.Layout(width="40px"))
    btn_prev  = widgets.Button(description="<", layout=widgets.Layout(width="40px"))
    btn_next  = widgets.Button(description=">", layout=widgets.Layout(width="40px"))
    btn_last  = widgets.Button(description=">|", layout=widgets.Layout(width="40px"))
    counter   = widgets.Label(value=f"{label}: {step_widget.value}/{step_widget.max}")

    def update_label(*_):
        counter.value = f"{label}: {step_widget.value}/{step_widget.max}"

    step_widget.observe(update_label, "value")
    step_widget.observe(update_label, "max")

    def on_first(_): step_widget.value = step_widget.min
    def on_prev(_):  step_widget.value = max(step_widget.min, step_widget.value - 1)
    def on_next(_):  step_widget.value = min(step_widget.max, step_widget.value + 1)
    def on_last(_):  step_widget.value = step_widget.max

    btn_first.on_click(on_first)
    btn_prev.on_click(on_prev)
    btn_next.on_click(on_next)
    btn_last.on_click(on_last)

    return widgets.HBox([btn_first, btn_prev, counter, btn_next, btn_last])


print("Helpers loaded (retina mode).")

## Rabin-Karp — Rolling Hash Search

### How it works

1. Map each character to a digit in base `d = |Σ|`
2. Compute the **fingerprint** (hash mod `q`) of the pattern and the first window of `T`
3. Slide the window: **remove** the leftmost digit, **shift** left, **add** the new rightmost digit — all mod `q`
4. On a **hash hit**, verify character-by-character (Las Vegas variant)

Step through the algorithm to see the rolling hash in action, including spurious hits and true matches.

In [ ]:
def char_to_digit(ch, alphabet):
    """Map character to digit in base d."""
    return alphabet.index(ch)


def rabin_karp_trace(T, P, q=101):
    """
    Step-by-step Rabin-Karp (Las Vegas).
    Returns list of snapshots capturing the full state at each window position.
    """
    alphabet = sorted(set(T + P))
    d = len(alphabet)
    n, m = len(T), len(P)
    if m > n:
        return [], alphabet, d

    h = pow(d, m - 1, q)  # d^(m-1) mod q

    # Pattern fingerprint via Horner
    p_hash = 0
    for ch in P:
        p_hash = (p_hash * d + char_to_digit(ch, alphabet)) % q

    # First window fingerprint
    t_hash = 0
    for j in range(m):
        t_hash = (t_hash * d + char_to_digit(T[j], alphabet)) % q

    snapshots = []
    matches = []

    for s in range(n - m + 1):
        hit = (t_hash == p_hash)
        verified = False
        is_match = False
        verify_steps = []

        if hit:
            # Character-by-character verification
            is_match = True
            for k in range(m):
                ok = T[s + k] == P[k]
                verify_steps.append((k, ok))
                if not ok:
                    is_match = False
                    break
            if is_match:
                matches.append(s)
            verified = True

        snapshots.append({
            "s": s,
            "t_hash": t_hash,
            "p_hash": p_hash,
            "hit": hit,
            "verified": verified,
            "is_match": is_match,
            "spurious": hit and not is_match,
            "verify_steps": verify_steps,
            "matches_so_far": list(matches),
            "h": h,
            "d": d,
            "q": q,
        })

        # Rolling hash update
        if s < n - m:
            old_digit = char_to_digit(T[s], alphabet)
            new_digit = char_to_digit(T[s + m], alphabet)
            t_hash = ((t_hash - old_digit * h) * d + new_digit) % q

    return snapshots, alphabet, d


def draw_rk_step(T, P, q, step_idx):
    """Draw a single Rabin-Karp step."""
    snapshots, alphabet, d = rabin_karp_trace(T, P, q)
    if not snapshots:
        return
    step_idx = min(step_idx, len(snapshots) - 1)
    snap = snapshots[step_idx]
    n, m = len(T), len(P)
    s = snap["s"]

    fig, axes = plt.subplots(3, 1, figsize=(max(n * 0.9, 10), 9.5),
                              gridspec_kw={"height_ratios": [3.5, 2.5, 2.5]})

    # ══════════════════════════════════════════════════════
    # Panel 1: T and P alignment with hash comparison
    # ══════════════════════════════════════════════════════
    ax = axes[0]
    ax.set_xlim(-2.5, n + 4.0)
    ax.set_ylim(-2.0, 4.5)
    ax.set_aspect("equal")
    ax.axis("off")

    # Index row
    for i in range(n):
        ax.text(i + 0.45, 3.8, str(i), ha="center", fontsize=8, color="#999")

    # T row
    t_hi = {}
    if snap["is_match"]:
        for k in range(m):
            t_hi[s + k] = COLORS["verified"]
    elif snap["spurious"]:
        last_verify = snap["verify_steps"][-1][0] if snap["verify_steps"] else 0
        for k, ok in snap["verify_steps"]:
            t_hi[s + k] = COLORS["match"] if ok else COLORS["spurious"]
    elif snap["hit"]:
        for k in range(m):
            t_hi[s + k] = COLORS["hash_match"]
    else:
        for k in range(m):
            t_hi[s + k] = COLORS["hash_miss"]
    draw_string_row(ax, 2.5, T, label="T", highlights=t_hi)

    # P row aligned at position s
    p_hi = {}
    if snap["is_match"]:
        for k in range(m):
            p_hi[k] = COLORS["verified"]
    elif snap["spurious"]:
        for k, ok in snap["verify_steps"]:
            p_hi[k] = COLORS["match"] if ok else COLORS["spurious"]
    elif snap["hit"]:
        for k in range(m):
            p_hi[k] = COLORS["hash_match"]
    else:
        for k in range(m):
            p_hi[k] = COLORS["hash_miss"]
    draw_string_row(ax, 0.9, P, label="P", highlights=p_hi, offset=s)

    # Window bracket below T
    ax.plot([s + 0.05, s + m - 0.15], [2.35, 2.35],
            color="#555", lw=2, solid_capstyle="round")

    # Hash values
    hash_color = COLORS["verified"] if snap["is_match"] else (
        COLORS["spurious"] if snap["spurious"] else (
        COLORS["hash_match"] if snap["hit"] else "#999"))

    ax.text(n + 1.0, 2.95, f"t_hash = {snap['t_hash']}",
            fontsize=11, fontweight="bold", color=hash_color, va="center")
    ax.text(n + 1.0, 1.35, f"p_hash = {snap['p_hash']}",
            fontsize=11, fontweight="bold", color="#555", va="center")

    # Comparison symbol
    if snap["hit"]:
        cmp_sym = "=="
        cmp_color = COLORS["match"]
    else:
        cmp_sym = "!="
        cmp_color = COLORS["mismatch"]
    ax.text(n + 1.0, 2.15, cmp_sym, fontsize=16, fontweight="bold",
            color=cmp_color, va="center")

    # Rolling hash formula annotation
    if s > 0:
        old_ch = T[s - 1]
        new_ch = T[s + m - 1] if s + m - 1 < n else "?"
        formula = (f"t_hash = (({snapshots[s-1]['t_hash']} - "
                   f"val('{old_ch}')·{snap['h']}) · {d} + "
                   f"val('{new_ch}')) mod {snap['q']}")
        ax.text(n / 2, -0.3, formula, ha="center", fontsize=9,
                color="#666", fontstyle="italic",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="#F5F5F5",
                          edgecolor="#CCC"))

    # Event description
    if snap["is_match"]:
        event = f"MATCH at position {s}! (hash hit + verified)"
        ev_color = COLORS["verified"]
    elif snap["spurious"]:
        fail_k = snap["verify_steps"][-1][0]
        event = f"SPURIOUS HIT at position {s} (hash matched, mismatch at P[{fail_k}])"
        ev_color = COLORS["spurious"]
    elif snap["hit"]:
        event = f"Hash hit at position {s} — verifying..."
        ev_color = COLORS["hash_match"]
    else:
        event = f"No hit at position {s} — skip"
        ev_color = "#999"

    ax.set_title(
        f"Rabin-Karp step {step_idx+1}/{len(snapshots)} | "
        f"s={s} | q={snap['q']} | d={d} | "
        f"matches={snap['matches_so_far']}\n{event}",
        fontsize=10, pad=8, color=ev_color)

    # ══════════════════════════════════════════════════════
    # Panel 2: Hash value timeline
    # ══════════════════════════════════════════════════════
    ax2 = axes[1]
    ax2.set_title("Hash values per window position (purple = p_hash)",
                  fontsize=10, pad=5, loc="left")

    positions = list(range(len(snapshots)))
    hashes = [sn["t_hash"] for sn in snapshots]
    bar_colors = []
    for i, sn in enumerate(snapshots):
        if i > step_idx:
            bar_colors.append("#E0E0E0")
        elif sn["is_match"]:
            bar_colors.append(COLORS["verified"])
        elif sn["spurious"]:
            bar_colors.append(COLORS["spurious"])
        elif sn["hit"]:
            bar_colors.append(COLORS["hash_match"])
        else:
            bar_colors.append(COLORS["hash_miss"])

    ax2.bar(positions, hashes, color=bar_colors, edgecolor="#777", linewidth=0.5,
            alpha=[1.0 if i <= step_idx else 0.3 for i in positions])

    # p_hash reference line
    ax2.axhline(y=snap["p_hash"], color=COLORS["border"], linestyle="--",
                linewidth=2, alpha=0.7, label=f"p_hash = {snap['p_hash']}")
    ax2.legend(fontsize=9, loc="upper right")

    # Highlight current bar
    if step_idx < len(positions):
        ax2.bar([step_idx], [hashes[step_idx]], color="none",
                edgecolor="#000", linewidth=2.5)

    ax2.set_xlabel("Window position s", fontsize=10)
    ax2.set_ylabel("t_hash", fontsize=10)
    ax2.set_xticks(positions)
    ax2.set_xticklabels([str(p) for p in positions], fontsize=8)

    # ══════════════════════════════════════════════════════
    # Panel 3: Alphabet mapping + parameters
    # ══════════════════════════════════════════════════════
    ax3 = axes[2]
    ax3.axis("off")
    ax3.set_xlim(0, 10)
    ax3.set_ylim(0, 3)

    # Alphabet mapping
    mapping_str = "  ".join([f"'{c}'→{i}" for i, c in enumerate(alphabet)])
    ax3.text(0.5, 2.5, f"Alphabet mapping (base d={d}):  {mapping_str}",
             fontsize=10, fontfamily="monospace", va="center")

    # Parameters
    ax3.text(0.5, 1.7, f"q = {snap['q']}    d = {d}    m = {m}    "
             f"h = d^(m-1) mod q = {snap['h']}",
             fontsize=10, fontfamily="monospace", va="center")

    # Stats
    total_hits = sum(1 for sn in snapshots[:step_idx+1] if sn["hit"])
    total_spurious = sum(1 for sn in snapshots[:step_idx+1] if sn["spurious"])
    total_matches = len(snap["matches_so_far"])
    ax3.text(0.5, 0.9,
             f"Hits so far: {total_hits}  |  "
             f"Spurious: {total_spurious}  |  "
             f"True matches: {total_matches}  |  "
             f"Positions checked: {step_idx+1}/{len(snapshots)}",
             fontsize=10, fontfamily="monospace", va="center",
             bbox=dict(boxstyle="round,pad=0.3", facecolor="#E8F5E9",
                       edgecolor="#A5D6A7"))

    safe_tight_layout(fig)
    plt.show()


# ── Interactive widgets ──────────────────────────────────────
T_rk = widgets.Text(value="abracadabra", description="T:",
                     layout=widgets.Layout(width="400px"))
P_rk = widgets.Text(value="abra", description="P:",
                     layout=widgets.Layout(width="400px"))
q_rk = widgets.IntSlider(value=101, min=2, max=997, step=1,
                          description="q (prime):",
                          layout=widgets.Layout(width="400px"))
step_rk = widgets.IntSlider(value=0, min=0, max=0, description="Step:",
                             layout=widgets.Layout(display="none"))


def _update_rk_max(*_):
    T, P = T_rk.value, P_rk.value
    q = q_rk.value
    if T and P and len(P) <= len(T):
        snaps, _, _ = rabin_karp_trace(T, P, q)
        step_rk.max = max(len(snaps) - 1, 0)
        step_rk.value = min(step_rk.value, step_rk.max)


T_rk.observe(_update_rk_max, "value")
P_rk.observe(_update_rk_max, "value")
q_rk.observe(_update_rk_max, "value")
_update_rk_max()


def _draw_rk(T, P, q, step):
    if T and P and len(P) <= len(T):
        draw_rk_step(T, P, q, step)


out_rk = widgets.interactive_output(_draw_rk,
    {"T": T_rk, "P": P_rk, "q": q_rk, "step": step_rk})
stepper_rk = make_stepper(step_rk, "Step")
display(T_rk, P_rk, q_rk, stepper_rk, out_rk)

## Effect of `q` on Spurious Hits

Choose a small `q` (e.g. 3, 5, 7) to see many spurious hits, or a large prime to see almost none. The bar chart shows, for each window position, whether the hash matched (hit) or not, and whether a hit was a true match or spurious.

In [ ]:
def draw_q_comparison(T, P, q_values):
    """Show side-by-side how different q values affect spurious hits."""
    fig, axes = plt.subplots(len(q_values), 1,
                              figsize=(max(len(T) * 0.6, 8), 3 * len(q_values)),
                              squeeze=False)

    for idx, q in enumerate(q_values):
        ax = axes[idx, 0]
        snapshots, alphabet, d = rabin_karp_trace(T, P, q)
        n_pos = len(snapshots)
        if n_pos == 0:
            continue

        colors = []
        labels_map = {"true match": COLORS["verified"],
                      "spurious hit": COLORS["spurious"],
                      "no hit": COLORS["hash_miss"]}
        categories = []
        for sn in snapshots:
            if sn["is_match"]:
                colors.append(COLORS["verified"])
                categories.append("true match")
            elif sn["spurious"]:
                colors.append(COLORS["spurious"])
                categories.append("spurious hit")
            else:
                colors.append(COLORS["hash_miss"])
                categories.append("no hit")

        positions = list(range(n_pos))
        hashes = [sn["t_hash"] for sn in snapshots]
        ax.bar(positions, hashes, color=colors, edgecolor="#777", linewidth=0.5)
        ax.axhline(y=snapshots[0]["p_hash"], color=COLORS["border"],
                    linestyle="--", linewidth=2, alpha=0.7)

        n_spurious = sum(1 for sn in snapshots if sn["spurious"])
        n_matches = sum(1 for sn in snapshots if sn["is_match"])
        ax.set_title(f"q = {q}  |  true matches: {n_matches}  |  "
                     f"spurious hits: {n_spurious}  |  "
                     f"total positions: {n_pos}",
                     fontsize=11, pad=6)
        ax.set_ylabel("hash", fontsize=10)
        if idx == len(q_values) - 1:
            ax.set_xlabel("Window position s", fontsize=10)
        ax.set_xticks(positions[::max(1, n_pos // 20)])

        # Legend
        from matplotlib.patches import Patch
        legend_elements = [
            Patch(facecolor=COLORS["verified"], edgecolor="#555", label="True match"),
            Patch(facecolor=COLORS["spurious"], edgecolor="#555", label="Spurious hit"),
            Patch(facecolor=COLORS["hash_miss"], edgecolor="#555", label="No hit"),
        ]
        ax.legend(handles=legend_elements, fontsize=8, loc="upper right")

    safe_tight_layout(fig)
    plt.show()


T_cmp = widgets.Text(value="abracadabracadabra", description="T:",
                      layout=widgets.Layout(width="400px"))
P_cmp = widgets.Text(value="abra", description="P:",
                      layout=widgets.Layout(width="400px"))
out_cmp = widgets.Output()


def _refresh_cmp(*_):
    T, P = T_cmp.value, P_cmp.value
    if T and P and len(P) <= len(T):
        with out_cmp:
            clear_output(wait=True)
            draw_q_comparison(T, P, [3, 7, 13, 101])

T_cmp.observe(_refresh_cmp, "value")
P_cmp.observe(_refresh_cmp, "value")
display(T_cmp, P_cmp, out_cmp)
_refresh_cmp()